# Environment Setup

This notebook produces a stand in for the Layer 1 demand forecasting model so that Layer 2 can be developed and demonstrated without waiting for the real model. It mimics the agreed Layer 1 output contract exactly: forecast demand per drug per facility for the next month, an adaptive buffer, uncertainty bounds, and the current stock position. Every row is marked is_mock so the table can be swapped for the real forecaster output without touching Layer 2.

The mock logic is intentionally simple but structurally faithful: condition case estimates from Layer 0 are projected one month ahead and translated into drug units through the condition to drug knowledge base, then inflated by a rule based buffer that mirrors the proposal formula of lead time times seasonality times accessibility.

## Drive Mount and Base Path

In [1]:
"""Mount Google Drive when running on Colab. Outside Colab the notebook falls
back to a local directory so the full pipeline can be tested end to end."""

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')

    """Base path untuk Google Drive storage"""
    base_path = '/content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton'
except ModuleNotFoundError:
    base_path = os.environ.get(
        'MATERNALINK_DATA_ROOT',
        os.path.abspath(os.path.join(os.getcwd(), '..', 'drive_local')),
    )

DATA_DIRS = {
    name: os.path.join(base_path, 'data', name)
    for name in [
        'master', 'transactional', 'layer0_output',
        'layer1_mock', 'layer2_output', 'audio_demo', 'eval',
    ]
}
for path in DATA_DIRS.values():
    os.makedirs(path, exist_ok=True)

print(f'Base path: {base_path}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base path: /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton


# Load Inputs

The mock consumes the Layer 0 contract table, the master data, the simulation case share parameters, and the latest stock position.

In [2]:
"""Load Layer 0 output, master data, and simulation parameters."""

import json
import math

import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

RAINY_MONTHS = {10, 11, 12, 1, 2, 3}
FORECAST_PERIOD = pd.Timestamp('2026-01-01')


def read_csv(folder, filename):
    df = pd.read_csv(os.path.join(DATA_DIRS[folder], filename))
    for col in ('period', 'forecast_period'):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])
    return df


facilities = read_csv('master', 'facilities.csv')
drugs = read_csv('master', 'drugs.csv')
kb_condition_drug = read_csv('master', 'kb_condition_drug.csv')
context_monthly = read_csv('transactional', 'context_monthly.csv')
stock_monthly = read_csv('transactional', 'stock_monthly.csv')
l0_condition_estimates = read_csv('layer0_output', 'l0_condition_estimates.csv')

with open(os.path.join(DATA_DIRS['eval'], 'sim_params.json')) as f:
    sim_params = json.load(f)

CASE_SHARE_OVERRIDES = {
    tuple(key.split('|')): value
    for key, value in sim_params['case_share_overrides'].items()
}
DEFAULT_CASE_SHARE = {int(k): v for k, v in sim_params['default_case_share'].items()}


def case_share(condition_id, drug_id, priority):
    """Fraction of treated cases of a condition that consume a given drug."""
    return CASE_SHARE_OVERRIDES.get((condition_id, drug_id), DEFAULT_CASE_SHARE[priority])


LAST_PERIOD = l0_condition_estimates['period'].max()
print(f'history ends {LAST_PERIOD.date()}, forecasting {FORECAST_PERIOD.date()}')

history ends 2025-12-01, forecasting 2026-01-01


# Mock Forecast Engine

## Condition Case Projection

Expected cases for the forecast month are the mean of the last three months of estimated total cases per facility and condition, with a seasonal uplift for the rainy sensitive conditions since the forecast month falls in the rainy season.

In [3]:
"""Project condition cases one month ahead from the Layer 0 contract table."""

recent = l0_condition_estimates[
    l0_condition_estimates['period'] > LAST_PERIOD - pd.DateOffset(months=3)
]
projected_cases = (
    recent.groupby(['facility_id', 'condition_id'])['estimated_total_cases']
    .mean()
    .reset_index(name='projected_cases')
)

forecast_is_rainy = FORECAST_PERIOD.month in RAINY_MONTHS
RAINY_SENSITIVE = {'K01': 1.3, 'K11': 1.2, 'K05': 1.05}
if forecast_is_rainy:
    projected_cases['projected_cases'] *= (
        projected_cases['condition_id'].map(RAINY_SENSITIVE).fillna(1.0)
    )

print(f'{len(projected_cases)} facility condition projections')
projected_cases.head()

330 facility condition projections


,facility_id,condition_id,projected_cases
0,PKM-001,K01,4.2042
1,PKM-001,K02,1.9710
2,PKM-001,K03,1.0000
3,PKM-001,K04,1.0000
4,PKM-001,K05,1.0500


## Cases to Drug Demand Translation

Projected cases are converted to drug units through the knowledge base using the same mechanistic formula as the rest of the system: cases times case share times daily dose times treatment duration, with trimester specific drugs scaled by the latest cohort composition.

In [4]:
"""Translate projected condition cases into forecast drug demand."""

drug_lookup = drugs.set_index('drug_id')
latest_context = (
    context_monthly[context_monthly['period'] == context_monthly['period'].max()]
    .set_index('facility_id')
)

demand_acc = {}
for case in projected_cases.itertuples():
    ctx = latest_context.loc[case.facility_id]
    total_pregnant = max(1, int(ctx.pregnant_t1 + ctx.pregnant_t2 + ctx.pregnant_t3))
    tri_shares = {
        'T1': ctx.pregnant_t1 / total_pregnant,
        'T2': ctx.pregnant_t2 / total_pregnant,
        'T3': ctx.pregnant_t3 / total_pregnant,
    }
    for kb in kb_condition_drug[kb_condition_drug['condition_id'] == case.condition_id].itertuples():
        share = case_share(kb.condition_id, kb.drug_id, kb.priority)
        trimesters = kb.trimester_applicable.split(',')
        if len(trimesters) < 3:
            share *= sum(tri_shares[t] for t in trimesters)
        drug = drug_lookup.loc[kb.drug_id]
        units = case.projected_cases * share * drug.standard_daily_dose * drug.treatment_duration_days
        key = (case.facility_id, kb.drug_id)
        demand_acc[key] = demand_acc.get(key, 0.0) + units

full_index = pd.MultiIndex.from_product(
    [facilities['facility_id'], drugs['drug_id']], names=['facility_id', 'drug_id'],
)
forecast_demand = (
    pd.Series(demand_acc)
    .rename_axis(['facility_id', 'drug_id'])
    .reindex(full_index, fill_value=0.0)
    .round()
    .astype(int)
    .rename('forecast_demand')
    .reset_index()
)

"""Facilities without cold chain neither stock nor dispense cold chain drugs;
those patients are referred, so their forecast demand is zero by convention."""
cold_chain_drugs = set(drugs.loc[drugs['requires_cold_chain'], 'drug_id'])
no_cold_chain_facilities = set(facilities.loc[~facilities['has_cold_chain'], 'facility_id'])
incompatible = (
    forecast_demand['drug_id'].isin(cold_chain_drugs)
    & forecast_demand['facility_id'].isin(no_cold_chain_facilities)
)
forecast_demand.loc[incompatible, 'forecast_demand'] = 0

print(f'total forecast demand {forecast_demand.forecast_demand.sum()} units')
forecast_demand.head()

total forecast demand 15026 units


,facility_id,drug_id,forecast_demand
0,PKM-001,OBT-001,40
1,PKM-001,OBT-002,5
2,PKM-001,OBT-003,29
3,PKM-001,OBT-004,145
4,PKM-001,OBT-005,18


## Adaptive Buffer Rule Engine

The buffer mirrors the proposal formula: a base buffer inflated by lead time, by rainy season access degradation, and by poor accessibility. The result is clipped to a fifty percent ceiling.

In [5]:
"""Rule based adaptive buffer per facility, applied uniformly across drugs."""

ACCESS_DEGRADATION = {'normal': 0.0, 'disrupted': 0.08, 'cut_off': 0.15}


def buffer_pct(fac):
    """Base buffer plus lead time, season, and accessibility components."""
    pct = 0.10
    pct += 0.015 * fac.lead_time_days
    if forecast_is_rainy:
        pct += ACCESS_DEGRADATION[fac.rainy_season_access]
    pct += 0.05 * (3 - fac.accessibility_score)
    return float(min(0.50, round(pct, 3)))


facility_buffer = pd.DataFrame([
    {'facility_id': fac.facility_id, 'buffer_pct': buffer_pct(fac)}
    for fac in facilities.itertuples()
])
facility_buffer.describe().loc[['min', 'mean', 'max']]

,buffer_pct
min,0.135000
mean,0.296367
max,0.500000


## Forecast Table Assembly

Uncertainty bounds wrap the point forecast, the latest closing stock becomes the current position, and every row carries the is_mock marker so downstream consumers can recognize the stand in.

In [6]:
"""Assemble the mock Layer 1 forecast table."""

current_stock = (
    stock_monthly[stock_monthly['period'] == stock_monthly['period'].max()]
    [['facility_id', 'drug_id', 'closing_stock']]
    .rename(columns={'closing_stock': 'current_stock'})
)

l1_forecast_mock = (
    forecast_demand
    .merge(facility_buffer, on='facility_id')
    .merge(current_stock, on=['facility_id', 'drug_id'], how='left')
)
l1_forecast_mock['current_stock'] = l1_forecast_mock['current_stock'].fillna(0).astype(int)
l1_forecast_mock['forecast_period'] = FORECAST_PERIOD
l1_forecast_mock['buffer_units'] = (
    l1_forecast_mock['forecast_demand'] * l1_forecast_mock['buffer_pct']
).apply(math.ceil)
l1_forecast_mock['total_requirement'] = (
    l1_forecast_mock['forecast_demand'] + l1_forecast_mock['buffer_units']
)
l1_forecast_mock['demand_lower'] = (l1_forecast_mock['forecast_demand'] * 0.8).round().astype(int)
l1_forecast_mock['demand_upper'] = (l1_forecast_mock['forecast_demand'] * 1.25).round().astype(int)
l1_forecast_mock['is_mock'] = True

l1_forecast_mock = l1_forecast_mock[[
    'facility_id', 'forecast_period', 'drug_id', 'forecast_demand',
    'buffer_pct', 'buffer_units', 'total_requirement', 'current_stock',
    'demand_lower', 'demand_upper', 'is_mock',
]]
print(f'{len(l1_forecast_mock)} forecast rows')
l1_forecast_mock.head()

900 forecast rows


,facility_id,forecast_period,drug_id,forecast_demand,buffer_pct,buffer_units,total_requirement,current_stock,demand_lower,demand_upper,is_mock
0,PKM-001,2026-01-01,OBT-001,40,0.5,20,60,50,32,50,True
1,PKM-001,2026-01-01,OBT-002,5,0.5,3,8,6,4,6,True
2,PKM-001,2026-01-01,OBT-003,29,0.5,15,44,14,23,36,True
3,PKM-001,2026-01-01,OBT-004,145,0.5,73,218,36,116,181,True
4,PKM-001,2026-01-01,OBT-005,18,0.5,9,27,0,14,22,True


# IFK Stock Scenario

The IFK warehouse is stocked at sixty to eighty five percent of the district wide requirement per drug. Deliberate scarcity is what makes the Layer 2 equity behavior visible: when supply cannot cover everyone, the optimizer has to choose, and those choices are what the explanation engine justifies.

In [7]:
"""Generate the mock IFK stock position under deliberate scarcity."""

district_requirement = (
    l1_forecast_mock.groupby('drug_id')['total_requirement'].sum().reset_index()
)
district_requirement['available_stock'] = (
    district_requirement['total_requirement']
    * rng.uniform(0.60, 0.85, size=len(district_requirement))
).round().astype(int)

ifk_stock_mock = district_requirement[['drug_id', 'available_stock']]
coverage = ifk_stock_mock['available_stock'].sum() / max(1, district_requirement['total_requirement'].sum())
print(f'district level stock covers {coverage:.0%} of total requirement')
ifk_stock_mock.head()

district level stock covers 76% of total requirement


,drug_id,available_stock
0,OBT-001,1250
1,OBT-002,150
2,OBT-003,700
3,OBT-004,4682
4,OBT-005,525


# Persistence

In [8]:
"""Persist the mock Layer 1 tables."""


def write_csv(df, folder, filename):
    out = df.copy()
    for col in ('period', 'forecast_period'):
        if col in out.columns:
            out[col] = pd.to_datetime(out[col]).dt.strftime('%Y-%m-%d')
    path = os.path.join(DATA_DIRS[folder], filename)
    out.to_csv(path, index=False)
    print(f'wrote {len(out):>7} rows -> {path}')


write_csv(l1_forecast_mock, 'layer1_mock', 'l1_forecast_mock.csv')
write_csv(ifk_stock_mock, 'layer1_mock', 'ifk_stock_mock.csv')

wrote     900 rows -> /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton/data/layer1_mock/l1_forecast_mock.csv
wrote      30 rows -> /content/drive/MyDrive/Kuliah/Lomba/Obat Bumil - AI Asean Hackahton/data/layer1_mock/ifk_stock_mock.csv


# Mock Validation

Structural checks that the table honors the Layer 1 output contract before Layer 2 consumes it.

In [9]:
"""Contract checks for the mock forecast."""

assert (l1_forecast_mock['forecast_demand'] >= 0).all()
assert (l1_forecast_mock['buffer_units'] >= 0).all()
assert (l1_forecast_mock['total_requirement'] >= l1_forecast_mock['forecast_demand']).all()
assert (l1_forecast_mock['demand_lower'] <= l1_forecast_mock['forecast_demand']).all()
assert (l1_forecast_mock['demand_upper'] >= l1_forecast_mock['forecast_demand']).all()
assert l1_forecast_mock['is_mock'].all()
assert (ifk_stock_mock['available_stock'] >= 0).all()
assert len(l1_forecast_mock) == len(facilities) * len(drugs)

demand_by_drug = l1_forecast_mock.groupby('drug_id')['total_requirement'].sum()
shortage = (ifk_stock_mock.set_index('drug_id')['available_stock'] < demand_by_drug).mean()
print(f'share of drugs in shortage at IFK level: {shortage:.0%}')
print('mock validation checks passed')

share of drugs in shortage at IFK level: 100%
mock validation checks passed
